In [19]:
# Importations

import time
from enderscope import SerialUtils, Stage 
import serial
from math import *
import threading

In [20]:
# Variables

  # Variables modifiables

rectangle = [40,30] # en mm, x puis y
coordonees_initiales = [10,10] # coordonnées origine [gauche, bas] du carré lié à la position de la boîte de pétri
vitesse_deplacement = 30
epaisseur_totale = 2

  #  Variables fixes
largeur_extrusion = 5 # en mm (largeur_extrusion x épaisseur = V_seringue x Surface seringue / V_imprimante)
nombre_passage = min(ceil(rectangle[0] /(2 * largeur_extrusion)),ceil(rectangle[1] /(2 * largeur_extrusion)))  # arrondit à l'entier superieur 
epaisseur_une_couche = 2
nombre_couche = ceil(epaisseur_totale/epaisseur_une_couche)
coordonees_initiales_carre_2 = [coordonees_initiales[0]+rectangle[0], coordonees_initiales[1]] # Coordoones de l origine pour le carre 2
coordonees_initiales_carre_3 = [coordonees_initiales[0]+ 2 *rectangle[0], coordonees_initiales[1]]

decalage = [1,1] # Pour le pousse seringue A, décalage de 1 selon x et 1 selon y. Si centré, on en deduit les decalages des autres 

hauteur_impression = 10 # en mm, dépend en partie de la hauteur de la boîte de pétri

In [21]:
# Ports

ports = SerialUtils.serial_ports() # Liste des ports
print (ports) # Affiche la liste

port_pousse_seringue = ports[0] # A modifier en fonction du branchement
port_imprimante = ports[1] # A modifier en fonction du branchement

s = Stage(port_imprimante, 115200) # Connexion imprimante
pousse_seringue = serial.Serial(port= port_pousse_seringue, baudrate=115200, timeout=0.01, writeTimeout=1) # Connexion pousse seringue

['COM6', 'COM8']


In [22]:
# Message respectif à envoyer à un pousse seringue pour le débloquer

def message_depart(lettre_pousse_seringue):
    return(f"{lettre_pousse_seringue}\n".encode('utf8')) # on transforme la f string en b string avec la commande de fin encode


    #if lettre_pousse_seringue == "A" :  # Le '\n' correspond à la fin du message
       #return b"A\n" 
    #if lettre_pousse_seringue == "B":
      # return b"B\n"
    #if lettre_pousse_seringue == "C":
     #   return b"C\n"

In [23]:
# Message respectif à envoyer à un pousse seringue pour l'arrêter 

def message_arret(lettre_pousse_seringue):
    if lettre_pousse_seringue == "A" :  # Le '\n' correspond à la fin du message
        return b"S\n" 
    if lettre_pousse_seringue == "B":
        return b"T\n"
    if lettre_pousse_seringue == "C":
        return b"U\n"


In [24]:
# Déterminer la postition qui'il faut demadnder à l'imprimante en fonction du pousse seringue utilisé

def position(lettre_pousse_seringue, coordonnees):
    position_tube = []
    if lettre_pousse_seringue == "A":
        position_tube.append(coordonnees[0] - decalage[0])
        position_tube.append(coordonnees[1] - decalage[1])
        return position_tube
    
    if lettre_pousse_seringue == "B":
        position_tube.append(coordonnees[0] + decalage[0])
        position_tube.append(coordonnees[1] - decalage[1])
        return position_tube

    if lettre_pousse_seringue == "C":
        position_tube.append(coordonnees[0] + decalage[0])
        position_tube.append(coordonnees[1] + decalage[1])
        return position_tube

In [25]:
# Forme un carré

def carre(): # Position du coin (bas_gauche) du carre

    x_rectangle = rectangle[0]
    y_rectangle = rectangle[1]

    for i in range (nombre_passage):
        for j in range(nombre_couche):
            s.write_code(f"M203 X{vitesse_deplacement}")
            s.write_code(f"M203 Y{vitesse_deplacement}")
            s.move_axis('x', x_rectangle)
            s.move_axis ('y', y_rectangle)
            s.move_axis('x', -x_rectangle)
            s.move_axis('y', - (y_rectangle -largeur_extrusion))

    

            s.move_axis('x', largeur_extrusion)
        
            x_rectangle = x_rectangle - (2 * largeur_extrusion)
            y_rectangle = y_rectangle - (2*largeur_extrusion)

    s.write_code(f"M400")

In [26]:
def se_positionner (lettre_pousse_seringue, coordonnes_debut):
    return (s.move_absolute(position(lettre_pousse_seringue, coordonnes_debut)[0],position(lettre_pousse_seringue, coordonnes_debut)[1], hauteur_impression))

In [27]:
# On fait le focus

s.home()

# Tracer le premier carré avec le pousse seringue A
se_positionner('A', coordonees_initiales)  # Aller en bas a gauche du carre 
s.write_code(f"M400")                      # Permet d'attendre que le mouvement soit fini
pousse_seringue.write(message_depart('A'))
carre() 
pousse_seringue.write(message_arret('A'))

# Tracer le deuxième carré avec le pousse seringue B
se_positionner('A', coordonees_initiales_carre_2)
s.write_code(f"M400") 
pousse_seringue.write(message_depart('A'))
carre() 
pousse_seringue.write(message_arret('A'))

# Tracer le troisième carré avec le pousse seringue C
se_positionner('A', coordonees_initiales_carre_3)
s.write_code(f"M400") 
pousse_seringue.write(message_depart('A'))
carre() 
pousse_seringue.write(message_arret('A'))

2